In [1]:
import sys
import os
import pandas as pd
import getpass
import torch

from google.colab import userdata
openai_key = userdata.get('OPENROUTER_API_KEY')

In [2]:
!pip -q install backoff

In [3]:
!git clone https://github.com/bencejdanko/bert-tweeteval

sys.path.append(os.path.abspath('/content/bert-tweeteval/src'))

from download import download_and_split_dataset
from llm_eval import LLMEvaluator, PROMPT_1_MINIMAL, PROMPT_2_STRUCTURED, LABELS

Cloning into 'bert-tweeteval'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 129 (delta 77), reused 81 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 161.32 KiB | 5.20 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [4]:
train_df, val_df, test_df = download_and_split_dataset()
print(f"Test set size: {len(test_df)}")

README.md: 0.00B [00:00, ?B/s]

emotion/train-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

emotion/test-00000-of-00001.parquet:   0%|          | 0.00/105k [00:00<?, ?B/s]

emotion/validation-00000-of-00001.parque(…):   0%|          | 0.00/28.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/374 [00:00<?, ? examples/s]

Test set size: 1421


In [5]:
evaluator_qwen = LLMEvaluator(hf_model_name="Qwen/Qwen3-4B-Instruct-2507")
evaluator_qwen.load_hf_model()

Loading HF model: Qwen/Qwen3-4B-Instruct-2507 on cuda...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [6]:
res_qwen_p1 = evaluator_qwen.evaluate(test_df, "hf", PROMPT_1_MINIMAL)


Evaluating hf (Batch size 100):   0%|          | 0/15 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Evaluating hf (Batch size 100): 100%|██████████| 15/15 [00:35<00:00,  2.39s/it]


In [7]:
res_qwen_p2 = evaluator_qwen.evaluate(test_df, "hf", PROMPT_2_STRUCTURED)


Evaluating hf (Batch size 100): 100%|██████████| 15/15 [01:10<00:00,  4.72s/it]


In [8]:
evaluator_openai = LLMEvaluator(openai_api_key=openai_key)

In [9]:
res_gpt_p1 = evaluator_openai.evaluate(test_df, "openai", PROMPT_1_MINIMAL)


Evaluating openai (Batch size 100): 100%|██████████| 15/15 [01:05<00:00,  4.38s/it]


In [10]:
res_gpt_p2 = evaluator_openai.evaluate(test_df, "openai", PROMPT_2_STRUCTURED)


Evaluating openai (Batch size 100): 100%|██████████| 15/15 [01:01<00:00,  4.13s/it]


In [14]:
results = {
    "GPT-4o-mini (Minimal)": res_gpt_p1,
    "GPT-4o-mini (Structured)": res_gpt_p2,
    "Qwen3-4B-Instruct-2507 (Minimal)": res_qwen_p1,
    "Qwen3-4B-Instruct-2507 (Structured)": res_qwen_p2
}

comparison_df = pd.DataFrame({
    k: {m: v[m] for m in ["Accuracy", "Macro-F1", "Time_per_100"]}
    for k, v in results.items()
}).transpose()

comparison_df

,Accuracy,Macro-F1,Time_per_100
GPT-4o-mini (Minimal),0.795918,0.597518,4.636414
GPT-4o-mini (Structured),0.821956,0.780503,4.374958
Qwen3-4B-Instruct-2507 (Minimal),0.751583,0.584864,2.614831
Qwen3-4B-Instruct-2507 (Structured),0.812104,0.758003,5.039879


In [16]:
comparison_df.to_markdown()

'|                                     |   Accuracy |   Macro-F1 |   Time_per_100 |\n|:------------------------------------|-----------:|-----------:|---------------:|\n| GPT-4o-mini (Minimal)               |   0.795918 |   0.597518 |        4.63641 |\n| GPT-4o-mini (Structured)            |   0.821956 |   0.780503 |        4.37496 |\n| Qwen3-4B-Instruct-2507 (Minimal)    |   0.751583 |   0.584864 |        2.61483 |\n| Qwen3-4B-Instruct-2507 (Structured) |   0.812104 |   0.758003 |        5.03988 |'

In [15]:
from datasets import Dataset
import huggingface_hub

hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(token=hf_token)

predictions_data = {
    'True_Labels': results["GPT-4o-mini (Minimal)"]["True_Labels"]
}
for k, v in results.items():
    predictions_data[k] = v["Predictions"]

predictions_dataset = Dataset.from_dict(predictions_data)
predictions_dataset.push_to_hub('bdanko/tweeteval-llm-evaluation-predictions')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.99kB / 5.99kB            

CommitInfo(commit_url='https://huggingface.co/datasets/bdanko/tweeteval-llm-evaluation-predictions/commit/e8f143dff03f2b2d5d1c3e4458c28dc545ba7349', commit_message='Upload dataset', commit_description='', oid='e8f143dff03f2b2d5d1c3e4458c28dc545ba7349', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bdanko/tweeteval-llm-evaluation-predictions', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bdanko/tweeteval-llm-evaluation-predictions'), pr_revision=None, pr_num=None)